# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [4]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one_model_60_trials_data_type_feat_eng_target_encoding.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one_model_60_trials_data_type_feat_eng_target_encoding.parquet')

In [5]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.001638,0.000283,0.998079,0.001796,0.000363,0.997841
1,0.996015,0.000339,0.003646,0.997132,0.000288,0.002580
2,0.009099,0.001175,0.989726,0.005254,0.000682,0.994064
3,0.003855,0.002145,0.994000,0.004080,0.001686,0.994234
4,0.995730,0.000018,0.004252,0.993613,0.000010,0.006377


In [6]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.013297,0.004871,0.981832,0.011730,0.004806,0.983465
1,0.407736,0.001829,0.590435,0.492656,0.003306,0.504038
2,0.998801,0.001053,0.000147,0.998502,0.001166,0.000333
3,0.976600,0.000013,0.023387,0.988312,0.000014,0.011675
4,0.004873,0.002234,0.992892,0.007801,0.002434,0.989766


# Machine Learning

In [7]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [8]:
def objective(trial, X, y):

    w0 = trial.suggest_float('weight_class_0', 0.001, 10.0, log=True)
    w1 = trial.suggest_float('weight_class_1', 0.001, 10.0, log=True)
    w2 = trial.suggest_float('weight_class_2', 0.001, 10.0, log=True)
    weights = np.array([w0, w1, w2])

    proba = X.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']].to_numpy()
    
    weighted_probas = proba * weights
    pred = np.argmax(weighted_probas, axis=1)
    
    score = balanced_accuracy_score(y, pred)

    return score


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=500, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-13 11:37:23,153] A new study created in memory with name: no-name-bfa5ee06-0a4a-4dd6-ae0b-ef9cf94d8480
                                                                                                                                                                                                                   

[I 2026-07-13 11:37:23,303] Trial 0 finished with value: 0.8144581878444114 and parameters: {'weight_class_0': 7.043056947794508, 'weight_class_1': 0.6272229742606508, 'weight_class_2': 0.35468969785009963}. Best is trial 0 with value: 0.8144581878444114.
[I 2026-07-13 11:37:23,314] Trial 10 finished with value: 0.7180299036603275 and parameters: {'weight_class_0': 0.011671034164083272, 'weight_class_1': 3.602517705285598, 'weight_class_2': 0.004695408267093219}. Best is trial 0 with value: 0.8144581878444114.
[I 2026-07-13 11:37:23,315] Trial 11 finished with value: 0.9208543774446841 and parameters: {'weight_class_0': 0.27949725923119, 'weight_class_1': 8.271373223848721, 'weight_class_2': 0.3835593955355141}. Best is trial 11 with value: 0.9208543774446841.
[I 2026-07-13 11:37:23,319] Trial 8 finished with value: 0.8788120901142978 and parameters: {'weight_class_0': 0.09554163721988959, 'weight_class_1': 0.013022457965217879, 'weight_class_2': 1.514545433209645}. Best is trial 11 wi

Best trial: 25. Best value: 0.946901:   5%|██████▊                                                                                                                                | 25/500 [00:00<00:09, 50.92it/s]

[I 2026-07-13 11:37:23,466] Trial 12 finished with value: 0.8746926235779663 and parameters: {'weight_class_0': 0.19783412168013723, 'weight_class_1': 0.6749289198542965, 'weight_class_2': 0.005805075958735924}. Best is trial 2 with value: 0.9439309120027916.
[I 2026-07-13 11:37:23,476] Trial 14 finished with value: 0.8193317456280051 and parameters: {'weight_class_0': 0.021770723563765665, 'weight_class_1': 0.08947855502256837, 'weight_class_2': 8.860942476409798}. Best is trial 2 with value: 0.9439309120027916.
[I 2026-07-13 11:37:23,497] Trial 15 finished with value: 0.8714293397769889 and parameters: {'weight_class_0': 0.02373506277918494, 'weight_class_1': 0.10408606896787156, 'weight_class_2': 6.8291460926901895}. Best is trial 2 with value: 0.9439309120027916.
[I 2026-07-13 11:37:23,518] Trial 16 finished with value: 0.8638562271960214 and parameters: {'weight_class_0': 0.026815640375531833, 'weight_class_1': 0.13793876046235437, 'weight_class_2': 8.488176605661659}. Best is tri

[I 2026-07-13 11:37:23,683] Trial 26 finished with value: 0.944674757455887 and parameters: {'weight_class_0': 0.002009560950718271, 'weight_class_1': 0.01107019874110776, 'weight_class_2': 0.0932621750654652}. Best is trial 25 with value: 0.9469005485280201.
[I 2026-07-13 11:37:23,696] Trial 27 finished with value: 0.9456594329812112 and parameters: {'weight_class_0': 0.0014044991871407242, 'weight_class_1': 0.0147642734003822, 'weight_class_2': 0.07989904924331862}. Best is trial 25 with value: 0.9469005485280201.
[I 2026-07-13 11:37:23,706] Trial 28 finished with value: 0.9400099105066227 and parameters: {'weight_class_0': 0.006284387376577522, 'weight_class_1': 0.016764447844980608, 'weight_class_2': 0.07151843293735832}. Best is trial 25 with value: 0.9469005485280201.
[I 2026-07-13 11:37:23,727] Trial 29 finished with value: 0.8181180391048923 and parameters: {'weight_class_0': 0.4362267970132112, 'weight_class_1': 0.017856856769830173, 'weight_class_2': 0.1327790521838177}. Best

...

[I 2026-07-13 11:37:31,839] Trial 468 finished with value: 0.9478449834714898 and parameters: {'weight_class_0': 0.0014734343152116751, 'weight_class_1': 0.020462109490735313, 'weight_class_2': 0.04664967953740525}. Best is trial 194 with value: 0.9496918515834345.
[I 2026-07-13 11:37:31,876] Trial 469 finished with value: 0.9492575713715284 and parameters: {'weight_class_0': 0.0017849893741630034, 'weight_class_1': 0.018361279717232445, 'weight_class_2': 0.017466629101408577}. Best is trial 194 with value: 0.9496918515834345.
[I 2026-07-13 11:37:31,884] Trial 470 finished with value: 0.9491690038064903 and parameters: {'weight_class_0': 0.001747846351479465, 'weight_class_1': 0.017568026917662953, 'weight_class_2': 0.016034321066200292}. Best is trial 194 with value: 0.9496918515834345.
[I 2026-07-13 11:37:31,904] Trial 471 finished with value: 0.9488399364228192 and parameters: {'weight_class_0': 0.0014321671513650588, 'weight_class_1': 0.012276350120382458, 'weight_class_2': 0.01722

Best trial: 194. Best value: 0.949692:  98%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████   | 489/500 [00:09<00:00, 53.84it/s]

[I 2026-07-13 11:37:32,068] Trial 478 finished with value: 0.9487256068957266 and parameters: {'weight_class_0': 0.0010047334834619606, 'weight_class_1': 0.02439220090835415, 'weight_class_2': 0.02403541204477487}. Best is trial 194 with value: 0.9496918515834345.
[I 2026-07-13 11:37:32,073] Trial 480 finished with value: 0.9363581465227808 and parameters: {'weight_class_0': 0.004790037643721256, 'weight_class_1': 0.012195050348501208, 'weight_class_2': 0.022671605314837855}. Best is trial 194 with value: 0.9496918515834345.
[I 2026-07-13 11:37:32,115] Trial 481 finished with value: 0.945547065267485 and parameters: {'weight_class_0': 0.004742417289742555, 'weight_class_1': 0.02496388881490001, 'weight_class_2': 0.026416320928776556}. Best is trial 194 with value: 0.9496918515834345.
[I 2026-07-13 11:37:32,119] Trial 482 finished with value: 0.9470529904532617 and parameters: {'weight_class_0': 0.001006589970040571, 'weight_class_1': 0.05310376353990562, 'weight_class_2': 0.02382688767

Best trial: 194. Best value: 0.949692: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [00:09<00:00, 54.14it/s]

[I 2026-07-13 11:37:32,261] Trial 490 finished with value: 0.9482507947530253 and parameters: {'weight_class_0': 0.0013707394067113063, 'weight_class_1': 0.05087916516556358, 'weight_class_2': 0.030276947519034828}. Best is trial 194 with value: 0.9496918515834345.
[I 2026-07-13 11:37:32,294] Trial 491 finished with value: 0.9483020005102443 and parameters: {'weight_class_0': 0.0014219184938814254, 'weight_class_1': 0.049679780304126696, 'weight_class_2': 0.03205379014571665}. Best is trial 194 with value: 0.9496918515834345.
[I 2026-07-13 11:37:32,302] Trial 492 finished with value: 0.9478741829775142 and parameters: {'weight_class_0': 0.0014076961342448927, 'weight_class_1': 0.008537952698106425, 'weight_class_2': 0.014209198350879852}. Best is trial 194 with value: 0.9496918515834345.
[I 2026-07-13 11:37:32,321] Trial 493 finished with value: 0.9482164170519249 and parameters: {'weight_class_0': 0.0013739592002654023, 'weight_class_1': 0.05173398533262669, 'weight_class_2': 0.030355

In [9]:
weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = X_test.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']].to_numpy() * weights

pred = np.argmax(weighted_probas, axis=1)

In [10]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [11]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.3_xgboost_submission.csv', index=False)

In [12]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
